In [29]:
import pandas as pd

customers = pd.read_excel("../data/retail_sales_dataset.xlsx", sheet_name="Customers")
products  = pd.read_excel("../data/retail_sales_dataset.xlsx", sheet_name="Products")
stores    = pd.read_excel("../data/retail_sales_dataset.xlsx", sheet_name="Stores")
transactions = pd.read_excel("../data/retail_sales_dataset.xlsx", sheet_name="Transactions")
#to change the fake city names to real city names for the project
dim_customer = pd.read_csv("DimCustomer.csv")
dim_store = pd.read_csv("DimStore.csv")

In [30]:
#Building fact table: products sales

sales_enriched = transactions.merge(
    products[["ProductID", "Category", "SubCategory", "UnitPrice", "CostPrice"]],
    on="ProductID",
    how="left"
)
sales_enriched.head()

,TransactionID,Date,CustomerID,ProductID,StoreID,Quantity,Discount,PaymentMethod,Category,SubCategory,UnitPrice,CostPrice
0,T00001,2024-06-18,C160,P014,S003,1,0.10,Bank Transfer,Fashion,Clothing,1342.75,797.94
1,T00002,2023-11-02,C171,P030,S004,3,0.15,Bank Transfer,Fashion,Watches,29.24,15.28
2,T00003,2024-03-28,C142,P002,S002,2,0.15,Mobile Money,Electronics,Television,818.76,527.62
3,T00004,2024-06-15,C174,P050,S002,5,0.10,Mobile Money,Fashion,Footwear,1044.64,775.07
4,T00005,2024-08-29,C141,P036,S001,3,0.10,Credit Card,Fashion,Watches,1501.46,1167.73


In [31]:
# Revenue after discount
sales_enriched["Revenue"] = (
    sales_enriched["Quantity"]
    * sales_enriched["UnitPrice"]
    * (1 - sales_enriched["Discount"])
)

# Cost
sales_enriched["Cost"] = (
    sales_enriched["Quantity"]
    * sales_enriched["CostPrice"]
)

# Profit
sales_enriched["Profit"] = (
    sales_enriched["Revenue"]
    - sales_enriched["Cost"]
)
sales_enriched.head()
sales_enriched.columns.tolist()

['TransactionID',
 'Date',
 'CustomerID',
 'ProductID',
 'StoreID',
 'Quantity',
 'Discount',
 'PaymentMethod',
 'Category',
 'SubCategory',
 'UnitPrice',
 'CostPrice',
 'Revenue',
 'Cost',
 'Profit']

In [32]:
# US Locations to replace fake city names (for DimStore and DimCustomer)

us_locations = [
    ("New York", "NY"),
    ("Los Angeles", "CA"),
    ("Chicago", "IL"),
    ("Houston", "TX"),
    ("Phoenix", "AZ"),
    ("Philadelphia", "PA"),
    ("San Antonio", "TX"),
    ("San Diego", "CA"),
    ("Dallas", "TX"),
    ("San Jose", "CA"),
    ("Austin", "TX"),
    ("Seattle", "WA"),
    ("Denver", "CO"),
    ("Boston", "MA"),
    ("Atlanta", "GA")
]

In [33]:

# Mapping fake city names to real US city names

all_fake_cities = sorted(
    set(dim_customer["City"].unique()).union(
        dim_store["City"].unique()
    )
)

city_mapping = {
    fake: us_locations[i % len(us_locations)]
    for i, fake in enumerate(all_fake_cities)
}

In [34]:
# Applying City Mapping

dim_customer[["City", "State"]] = (
    dim_customer["City"]
    .map(city_mapping)
    .apply(pd.Series)
)
dim_store[["City", "State"]] = (
    dim_store["City"]
    .map(city_mapping)
    .apply(pd.Series)
)

In [35]:
# Adding Country column

dim_customer["Country"] = "USA"
dim_store["Country"] = "USA"


In [36]:
# Test the changes of city names

print(dim_customer.head(3))
print(dim_store.head())


  CustomerID FirstName LastName Gender   BirthDate     City    JoinDate  Age  \
0       C001   Michael    Davis      M  1996-09-11   Austin  2022-09-25   29   
1       C002   Michael   Miller      M  1959-08-18  Phoenix  2020-11-03   66   
2       C003     Carol     Hays      F  2005-04-19  Atlanta  2024-02-12   20   

  State Country  
0    TX     USA  
1    AZ     USA  
2    GA     USA  
  StoreID                StoreName      City Region State Country
0    S001  MegaMart Jimenezborough    Denver  South    CO     USA
1    S002       MegaMart Peckmouth   Seattle   East    WA     USA
2    S003     MegaMart New Michele  New York   West    NY     USA
3    S004     MegaMart Brianahaven   Atlanta  North    GA     USA
4    S005       MegaMart Johnmouth   Atlanta   East    GA     USA


In [37]:
#Adding Customer demographics
sales_enriched = sales_enriched.merge(
    customers[["CustomerID", "Gender", "BirthDate", "City", "JoinDate"]],
    on="CustomerID",
    how="left"
)
sales_enriched.head()
sales_enriched.columns.tolist()

['TransactionID',
 'Date',
 'CustomerID',
 'ProductID',
 'StoreID',
 'Quantity',
 'Discount',
 'PaymentMethod',
 'Category',
 'SubCategory',
 'UnitPrice',
 'CostPrice',
 'Revenue',
 'Cost',
 'Profit',
 'Gender',
 'BirthDate',
 'City',
 'JoinDate']

In [38]:
#Adding date features (Year / Month / Quarter / Age)

from datetime import datetime

# Ensure datetime
sales_enriched["Date"] = pd.to_datetime(sales_enriched["Date"])
sales_enriched["BirthDate"] = pd.to_datetime(sales_enriched["BirthDate"])
sales_enriched["JoinDate"] = pd.to_datetime(sales_enriched["JoinDate"])

# Calendar features
sales_enriched["Year"] = sales_enriched["Date"].dt.year
sales_enriched["Month"] = sales_enriched["Date"].dt.month
sales_enriched["MonthName"] = sales_enriched["Date"].dt.strftime("%B")
sales_enriched["Quarter"] = sales_enriched["Date"].dt.quarter
sales_enriched["YearMonth"] = sales_enriched["Date"].dt.to_period("M").astype(str)

# Age at time of transaction
sales_enriched["CustomerAge"] = (
    (sales_enriched["Date"] - sales_enriched["BirthDate"]).dt.days // 365
)

# Tenure in days since join date
sales_enriched["CustomerTenureDays"] = (
    (sales_enriched["Date"] - sales_enriched["JoinDate"]).dt.days
)

In [39]:
# Building fact table: products sales with customer demographics and date features

fact_sales = sales_enriched[[
    "TransactionID",
    "Date",
    "CustomerID",
    "ProductID",
    "StoreID",
    "Quantity",
    "Discount",
    "Revenue",
    "Cost",
    "Profit"
]]
# Optional: Keep Year, Month, YearMonth in the fact table if you don't want a separate date dimension yet


In [40]:
#DimCustomer table

dim_customer = dim_customer.copy()
dim_customer["BirthDate"] = pd.to_datetime(dim_customer["BirthDate"])
dim_customer["JoinDate"] = pd.to_datetime(dim_customer["JoinDate"])

# Current age (for reference, based on 'today' of the dataset, e.g. 2025-12-31)
ref_date = pd.to_datetime("2025-12-31")
dim_customer["Age"] = (ref_date - dim_customer["BirthDate"]).dt.days // 365

In [41]:
# DimProduct(Category, SubCategory, UnitPrice, CostPrice)
dim_product = products.copy()
# Typical columns: ProductID, ProductName, Category, SubCategory, UnitPrice, CostPrice

In [42]:
# DimStore(Location, Region, etc.)
dim_store = dim_store.copy()
# Typical columns: StoreID, StoreName, City, Region, etc.

In [43]:
# DimDate table (Date, Year, Month, Quarter, etc.) Optional creation of date dimension table

date_dim = (
    sales_enriched[["Date"]]
    .drop_duplicates()
    .rename(columns={"Date": "DateKey"})
    .sort_values("DateKey")
)

date_dim["Year"] = date_dim["DateKey"].dt.year
date_dim["Month"] = date_dim["DateKey"].dt.month
date_dim["MonthName"] = date_dim["DateKey"].dt.strftime("%B")
date_dim["Quarter"] = date_dim["DateKey"].dt.quarter
date_dim["YearMonth"] = date_dim["DateKey"].dt.to_period("M").astype(str)

In [44]:
# Exporting to CSV files

fact_sales.to_csv("FactSales.csv", index=False)
dim_customer.to_csv("DimCustomer.csv", index=False)
dim_product.to_csv("DimProduct.csv", index=False)
dim_store.to_csv("DimStore.csv", index=False)
date_dim.to_csv("DimDate.csv", index=False)